# AATA Detection — Training

This notebook trains the `KneeClassifier` model on the internal DASA dataset
and evaluates it on the internal validation and test sets.
Grad-CAM visualizations are produced for selected true-positive, false-positive,
and false-negative cases.

## Configuration

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
IMAGE_FOLDER   = "/path/to/JPEG_NSeries0-25"   # directory of preprocessed JPEG slices
METADATA_CSV   = "/path/to/final_df.csv"        # slice-level metadata with fold assignments
WEIGHTS_OUTPUT = "weights/model.pth"            # destination for the best checkpoint
GRADCAM_DIR    = "outputs/gradcam/internal"     # output directory for Grad-CAM PDFs

# ── Data split ──────────────────────────────────────────────────────────────
TRAIN_FOLDS = [2, 3, 4]
VAL_FOLD    = [1]
TEST_FOLD   = [0]

# ── Normalization ───────────────────────────────────────────────────────────
# Precomputed from 500 randomly sampled training slices.
PIXEL_MEAN = 0.0028
PIXEL_STD  = 0.0290

# ── Training hyperparameters ────────────────────────────────────────────────
BATCH_SIZE    = 128
NUM_EPOCHS    = 30
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 1e-4
L1_LAMBDA     = 1e-5
LR_GAMMA      = 0.9    # exponential decay factor applied each epoch

# ── Evaluation ──────────────────────────────────────────────────────────────
THRESHOLD    = 0.1717  # study-level probability threshold, optimized on the validation set
N_BOOTSTRAP  = 1000
SEED         = 42

# ── Device ──────────────────────────────────────────────────────────────────
DEVICE = "cuda" if __import__('torch').cuda.is_available() else "cpu"
NUM_WORKERS = 16

## Imports

In [ ]:
import os
import random

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from matplotlib.backends.backend_pdf import PdfPages
from PIL import Image
from sklearn.metrics import f1_score, precision_recall_curve, auc, precision_score, recall_score
from torch.optim.lr_scheduler import ExponentialLR
from torch.utils.data import DataLoader
from tqdm.notebook import tqdm

from src.dataset import MRISliceDataset, build_transforms
from src.metrics import (
    bootstrap_metrics,
    bootstrap_roc,
    compute_metrics,
    plot_roc,
    print_metrics_table,
)
from src.model import KneeClassifier, generate_gradcam

torch.manual_seed(SEED)
device = torch.device(DEVICE)

## Data

In [ ]:
final_df = pd.read_csv(METADATA_CSV)

train_df = final_df[final_df["fold"].isin(TRAIN_FOLDS)]
val_df   = final_df[final_df["fold"].isin(VAL_FOLD)]
test_df  = final_df[final_df["fold"].isin(TEST_FOLD)]

print(f"Train slices : {len(train_df):,}")
print(f"Val slices   : {len(val_df):,}")
print(f"Test slices  : {len(test_df):,}")

In [ ]:
transform_train = build_transforms(PIXEL_MEAN, PIXEL_STD, augment=True)
transform_eval  = build_transforms(PIXEL_MEAN, PIXEL_STD, augment=False)

train_loader = DataLoader(
    MRISliceDataset(IMAGE_FOLDER, train_df, transform=transform_train),
    batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS,
)
val_loader = DataLoader(
    MRISliceDataset(IMAGE_FOLDER, val_df, transform=transform_eval),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
)
test_loader = DataLoader(
    MRISliceDataset(IMAGE_FOLDER, test_df, transform=transform_eval),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
)

## Training

In [ ]:
model     = KneeClassifier(num_classes=1).to(device)
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = ExponentialLR(optimizer, gamma=LR_GAMMA)

os.makedirs(os.path.dirname(WEIGHTS_OUTPUT), exist_ok=True)

history = {"train_loss": [], "val_loss": [], "train_f1": [], "val_f1": [],
           "train_precision": [], "val_precision": [], "train_recall": [], "val_recall": []}
best_val_f1 = 0.0

for epoch in range(NUM_EPOCHS):
    # ── Training phase ─────────────────────────────────────────────────────
    model.train()
    running_loss, all_labels, all_preds = 0.0, [], []

    for images, labels, _ in tqdm(train_loader, desc=f"Train [{epoch+1}/{NUM_EPOCHS}]"):
        images = images.to(device)
        labels = labels.float().unsqueeze(1).to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        l1_penalty = sum(p.abs().sum() for p in model.parameters())
        loss = loss + L1_LAMBDA * l1_penalty

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        all_labels.extend(labels.cpu().numpy())
        all_preds.extend((torch.sigmoid(outputs) > 0.5).detach().cpu().numpy())

    scheduler.step()
    history["train_loss"].append(running_loss / len(train_loader.dataset))
    history["train_f1"].append(f1_score(all_labels, all_preds))
    history["train_precision"].append(precision_score(all_labels, all_preds))
    history["train_recall"].append(recall_score(all_labels, all_preds))

    # ── Validation phase ───────────────────────────────────────────────────
    model.eval()
    running_loss, all_labels, all_preds = 0.0, [], []

    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc=f"Val   [{epoch+1}/{NUM_EPOCHS}]"):
            images = images.to(device)
            labels = labels.float().unsqueeze(1).to(device)
            outputs = model(images)
            running_loss += criterion(outputs, labels).item() * labels.size(0)
            all_labels.extend(labels.cpu().numpy())
            all_preds.extend((torch.sigmoid(outputs) > 0.5).cpu().numpy())

    val_f1 = f1_score(all_labels, all_preds)
    history["val_loss"].append(running_loss / len(val_loader.dataset))
    history["val_f1"].append(val_f1)
    history["val_precision"].append(precision_score(all_labels, all_preds))
    history["val_recall"].append(recall_score(all_labels, all_preds))

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), WEIGHTS_OUTPUT)
        print(f"  Checkpoint saved — val F1: {best_val_f1:.4f}")

    print(f"Epoch {epoch+1:02d}/{NUM_EPOCHS} | "
          f"train loss {history['train_loss'][-1]:.4f} f1 {history['train_f1'][-1]:.4f} | "
          f"val loss {history['val_loss'][-1]:.4f} f1 {history['val_f1'][-1]:.4f}")

In [ ]:
epochs = range(1, NUM_EPOCHS + 1)
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
pairs = [
    ("train_loss",      "val_loss",      "Loss"),
    ("train_f1",        "val_f1",        "F1-Score"),
    ("train_precision", "val_precision", "Precision"),
    ("train_recall",    "val_recall",    "Recall"),
]
for ax, (train_key, val_key, ylabel) in zip(axes.flat, pairs):
    ax.plot(epochs, history[train_key], label="Train")
    ax.plot(epochs, history[val_key],   label="Validation")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Training and Validation {ylabel}")
    ax.legend()
    ax.grid(True, linestyle=":", alpha=0.7)
plt.tight_layout()
plt.show()

## Validation Set — Threshold Optimization

In [ ]:
def run_inference(loader, model, device):
    """Return per-slice predictions as a list of dicts."""
    model.eval()
    records = []
    with torch.no_grad():
        for images, labels, sop_uids in tqdm(loader, desc="Inference"):
            images = images.to(device)
            probs = torch.sigmoid(model(images)).cpu().numpy()
            for uid, prob, lbl in zip(sop_uids, probs, labels.numpy()):
                records.append({"SOPInstanceUID": uid, "Probability": float(prob), "label": int(lbl)})
    return pd.DataFrame(records)


model.load_state_dict(torch.load(WEIGHTS_OUTPUT, map_location=device))

sop_val = run_inference(val_loader, model, device)
sop_val = sop_val.merge(final_df[["SOPInstanceUID", "StudyInstanceUID", "grupo"]], on="SOPInstanceUID")

study_val = (
    sop_val.groupby("StudyInstanceUID")
    .agg(Avg_Probability=("Probability", "mean"), grupo=("grupo", "first"))
    .reset_index()
)

In [ ]:
thresholds = np.linspace(0, 1, 200)
f1_scores  = [f1_score(study_val["grupo"], (study_val["Avg_Probability"] >= t).astype(int))
              for t in thresholds]

best_idx = int(np.argmax(f1_scores))
best_threshold = thresholds[best_idx]

plt.figure(figsize=(8, 4))
plt.plot(thresholds, f1_scores)
plt.axvline(best_threshold, color="red", linestyle="--",
            label=f"Best threshold = {best_threshold:.4f}")
plt.xlabel("Threshold")
plt.ylabel("F1-Score")
plt.title("Threshold Optimization (Validation Set)")
plt.legend()
plt.grid(True, linestyle=":", alpha=0.7)
plt.show()

print(f"Optimal threshold: {best_threshold:.4f}  (hardcoded as THRESHOLD = {THRESHOLD})")

## Internal Test Set Evaluation

In [ ]:
sop_test = run_inference(test_loader, model, device)
sop_test = sop_test.merge(final_df[["SOPInstanceUID", "StudyInstanceUID", "grupo"]], on="SOPInstanceUID")

study_test = (
    sop_test.groupby("StudyInstanceUID")
    .agg(Avg_Probability=("Probability", "mean"), grupo=("grupo", "first"))
    .reset_index()
)
study_test["Predicted"] = (study_test["Avg_Probability"] >= THRESHOLD).astype(int)

y_true_test  = study_test["grupo"].values
y_pred_test  = study_test["Predicted"].values
y_score_test = study_test["Avg_Probability"].values

In [ ]:
point_est = compute_metrics(y_true_test, y_pred_test)
ci         = bootstrap_metrics(y_true_test, y_pred_test, n_bootstraps=N_BOOTSTRAP, seed=SEED)

print("Internal Test Set — Point Estimates and 95% Bootstrap Confidence Intervals\n")
print_metrics_table(point_est, ci)

In [ ]:
plot_roc(y_true_test, y_score_test, title="ROC Curve — Internal Test",
         save_path="outputs/roc_internal.png")
plot_roc(y_true_test, y_score_test, title="ROC Curve — Internal Test (zoomed)",
         zoom=True, save_path="outputs/roc_internal_zoom.png")

In [ ]:
from sklearn.metrics import precision_recall_curve, auc as sklearn_auc

precision_curve, recall_curve, _ = precision_recall_curve(y_true_test, y_score_test)
pr_auc = sklearn_auc(recall_curve, precision_curve)

plt.figure(figsize=(6, 5))
plt.plot(recall_curve, precision_curve, color="steelblue", lw=2, label=f"PR AUC = {pr_auc:.4f}")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — Internal Test")
plt.legend(loc="lower left")
plt.grid(True, linestyle=":", alpha=0.7)
plt.tight_layout()
plt.savefig("outputs/pr_internal.png", dpi=300, bbox_inches="tight")
plt.show()

## Grad-CAM Visualization

In [ ]:
# Select the 5 true-positive and 5 false-positive studies with the highest
# predicted probability, plus all false-negative studies.
tp_studies = study_test[(study_test["grupo"] == 1) & (study_test["Predicted"] == 1)]
fp_studies = study_test[(study_test["grupo"] == 0) & (study_test["Predicted"] == 1)]
fn_studies = study_test[(study_test["grupo"] == 1) & (study_test["Predicted"] == 0)]

gradcam_studies = pd.concat([
    tp_studies.nlargest(5, "Avg_Probability"),
    fp_studies.nlargest(5, "Avg_Probability"),
    fn_studies,
]).drop_duplicates(subset="StudyInstanceUID")

print(f"Studies selected for Grad-CAM: {len(gradcam_studies)} "
      f"({len(tp_studies.nlargest(5, 'Avg_Probability'))} TP, "
      f"{len(fp_studies.nlargest(5, 'Avg_Probability'))} FP, "
      f"{len(fn_studies)} FN)")

In [ ]:
os.makedirs(GRADCAM_DIR, exist_ok=True)

model.load_state_dict(torch.load(WEIGHTS_OUTPUT, map_location=device))
model.eval()

target_layer = model.base_model.layer4
selected_sops = sop_test[sop_test["StudyInstanceUID"].isin(gradcam_studies["StudyInstanceUID"])]

for study_uid in tqdm(gradcam_studies["StudyInstanceUID"], desc="Grad-CAM"):
    study_sops = selected_sops[selected_sops["StudyInstanceUID"] == study_uid]
    true_label = gradcam_studies.loc[
        gradcam_studies["StudyInstanceUID"] == study_uid, "grupo"
    ].values[0]

    pdf_path = os.path.join(GRADCAM_DIR, f"{study_uid}.pdf")
    with PdfPages(pdf_path) as pdf:
        for _, row in study_sops.iterrows():
            image_path = os.path.join(IMAGE_FOLDER, f"{row['SOPInstanceUID']}.jpeg")
            image = Image.open(image_path).convert("RGB")
            input_tensor = transform_eval(image.convert("L")).unsqueeze(0).to(device)
            input_tensor = torch.from_numpy(np.repeat(input_tensor.cpu().numpy(), 3, axis=1)).to(device)

            cam = generate_gradcam(model, input_tensor, target_layer)
            cam_resized = cv2.resize(cam, (image.width, image.height), interpolation=cv2.INTER_CUBIC)

            fig, ax = plt.subplots(figsize=(7, 7))
            ax.imshow(image, cmap="gray")
            ax.imshow(cam_resized, cmap="jet", alpha=0.45)
            ax.set_title(
                f"True: {true_label} | Prob: {row['Probability']:.3f}",
                fontsize=13,
            )
            ax.axis("off")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)

print(f"Grad-CAM PDFs saved to {GRADCAM_DIR}")